In [6]:
import requests
import pandas as pd
from tqdm import tqdm
import datetime as dt
import glob
import os

In [7]:
CHI_df = pd.read_csv("C:/Users/qtg4ys/Documents/DS3001/DS4002/Crimes_-_2001_to_Present_20260311.csv", nrows=5)
print(CHI_df.columns.tolist())
print(CHI_df.dtypes)
print(CHI_df.head(2))

['ID', 'Case Number', 'Date', 'Block', 'IUCR', 'Primary Type', 'Description', 'Location Description', 'Arrest', 'Domestic', 'Beat', 'District', 'Ward', 'Community Area', 'FBI Code', 'X Coordinate', 'Y Coordinate', 'Year', 'Updated On', 'Latitude', 'Longitude', 'Location']
ID                        int64
Case Number              object
Date                     object
Block                    object
IUCR                     object
Primary Type             object
Description              object
Location Description     object
Arrest                     bool
Domestic                   bool
Beat                      int64
District                  int64
Ward                      int64
Community Area            int64
FBI Code                 object
X Coordinate              int64
Y Coordinate              int64
Year                      int64
Updated On               object
Latitude                float64
Longitude               float64
Location                 object
dtype: object
         

In [8]:
NYC_df = pd.read_csv("C:/Users/qtg4ys/Documents/DS3001/DS4002/NYPD_Complaint_Data_Historic_20260311 (1).csv", nrows=5)
print(NYC_df.columns.tolist())
print(NYC_df.dtypes)
print(NYC_df.head(2))

['CMPLNT_NUM', 'CMPLNT_FR_DT', 'CMPLNT_FR_TM', 'CMPLNT_TO_DT', 'CMPLNT_TO_TM', 'ADDR_PCT_CD', 'RPT_DT', 'KY_CD', 'OFNS_DESC', 'PD_CD', 'PD_DESC', 'CRM_ATPT_CPTD_CD', 'LAW_CAT_CD', 'BORO_NM', 'LOC_OF_OCCUR_DESC', 'PREM_TYP_DESC', 'JURIS_DESC', 'JURISDICTION_CODE', 'PARKS_NM', 'HADEVELOPT', 'HOUSING_PSA', 'X_COORD_CD', 'Y_COORD_CD', 'SUSP_AGE_GROUP', 'SUSP_RACE', 'SUSP_SEX', 'TRANSIT_DISTRICT', 'Latitude', 'Longitude', 'Lat_Lon', 'PATROL_BORO', 'STATION_NAME', 'VIC_AGE_GROUP', 'VIC_RACE', 'VIC_SEX']
CMPLNT_NUM             int64
CMPLNT_FR_DT          object
CMPLNT_FR_TM          object
CMPLNT_TO_DT          object
CMPLNT_TO_TM          object
ADDR_PCT_CD            int64
RPT_DT                object
KY_CD                  int64
OFNS_DESC             object
PD_CD                  int64
PD_DESC               object
CRM_ATPT_CPTD_CD      object
LAW_CAT_CD            object
BORO_NM               object
LOC_OF_OCCUR_DESC     object
PREM_TYP_DESC         object
JURIS_DESC            object
JURI

In [9]:
print(NYC_df["CMPLNT_FR_DT"].dropna().unique()[:20])

# Check what 2025 dates look like specifically
NYC_df["parsed"] = pd.to_datetime(NYC_df["CMPLNT_FR_DT"], errors="coerce")
print("\n2025 rows:", (NYC_df["parsed"].dt.year == 2025).sum())
print("NaT count:", NYC_df["parsed"].isna().sum())

['12/02/2024' '12/29/2024' '12/17/2024' '12/23/2024']

2025 rows: 0
NaT count: 0


In [10]:
NYC_curr_df = pd.read_csv("C:/Users/qtg4ys/Documents/DS3001/DS4002/NYPD_Complaint_Data_Current_(Year_To_Date)_20260311.csv"
, nrows=5)
print(NYC_curr_df.columns.tolist())
print(NYC_curr_df.dtypes)
print(NYC_curr_df.head(2))


['CMPLNT_NUM', 'ADDR_PCT_CD', 'BORO_NM', 'CMPLNT_FR_DT', 'CMPLNT_FR_TM', 'CMPLNT_TO_DT', 'CMPLNT_TO_TM', 'CRM_ATPT_CPTD_CD', 'HADEVELOPT', 'HOUSING_PSA', 'JURISDICTION_CODE', 'JURIS_DESC', 'KY_CD', 'LAW_CAT_CD', 'LOC_OF_OCCUR_DESC', 'OFNS_DESC', 'PARKS_NM', 'PATROL_BORO', 'PD_CD', 'PD_DESC', 'PREM_TYP_DESC', 'RPT_DT', 'STATION_NAME', 'SUSP_AGE_GROUP', 'SUSP_RACE', 'SUSP_SEX', 'TRANSIT_DISTRICT', 'VIC_AGE_GROUP', 'VIC_RACE', 'VIC_SEX', 'X_COORD_CD', 'Y_COORD_CD', 'Latitude', 'Longitude', 'Lat_Lon', 'New Georeferenced Column']
CMPLNT_NUM                    int64
ADDR_PCT_CD                   int64
BORO_NM                      object
CMPLNT_FR_DT                 object
CMPLNT_FR_TM                 object
CMPLNT_TO_DT                 object
CMPLNT_TO_TM                 object
CRM_ATPT_CPTD_CD             object
HADEVELOPT                   object
HOUSING_PSA                 float64
JURISDICTION_CODE             int64
JURIS_DESC                   object
KY_CD                         int64
L

In [11]:
# DATE WINDOW

START_DATE = pd.Timestamp("2022-01-01")
END_DATE   = pd.Timestamp("2025-12-31")

# OFFENSE CLASSIFICATION MAPS

#Charlottesville
CVILLE_VIOLENT = {
    "ABDUCTION / KID", "ABDUCTION OR KI", "ARMED ROBBERY", "ASSAULT",
    "ASSAULT / SEXUA", "BANK ROBBERY", "CARJACKING", "COMMON LAW ROBB",
    "COR - ABUSE / N", "COR - ABUSE OR", "COR - ABUSE OR ", "COR - ATTEMPTED",
    "COR - DOMESTIC", "COR - DOMESTIC ", "COR - INDECENT", "COR - INDECENT ",
    "CRISIS - VIOLEN", "CRISIS WITH WEA", "DOMESTIC DIST /", "DOMESTIC VIOLEN",
    "DOMESTIC WITH A", "FIGHT", "GUNSHOT WOUND", "GUNSHOT WOUND N",
    "HOME INVASION", "HOSTAGE", "INDECENCY OR  L", "INDECENT",
    "RAPE KIT", "RAPE NO EMS", "ROBBERY / CARJA", "ROBBERY IN PROG",
    "SEXUAL ASSAULT", "SHOOT/DRIVE BY", "SHOOT/DRIVE BY ", "STABBING",
    "STABBING NO EMS", "STALKING", "THREATS", "WEAPONS / FIREA", "WEAPONS VIOLATI",
}

CVILLE_PROPERTY = {
    "BREAK AND ENTER", "BREAK IN", "BREAK IN IN PRO", "BREAK IN OR LAR",
    "BURGLARY", "COUNTERFEITING/", "DAMAGE / VANDAL", "DAMAGE TO PROPE",
    "EMBEZZLEMENT", "FRAUD", "FRAUD - CONFIDE", "FRAUD - CREDIT ",
    "FRAUD - FALSE P", "FRAUD - IMPERSO", "FRAUD - WIRE/CO", "FRAUD - WORTHLE",
    "FRAUD / DECEPTI", "FRAUD IN PROGRE", "FRAUD-HACKING/C", "FRAUD-IDENTITY ",
    "LARCENY", "LARCENY - ALL O", "LARCENY - AUTOM", "LARCENY - FROM ",
    "LARCENY - PURSE", "LARCENY - SHOPL", "LARCENY IN PROG", "LARCENY OF VEH",
    "LARCENY OF VEH ", "LARCENY OF VEHI", "MISCHIEF", "SHOPLIFTER",
    "STOLEN VEHICLE", "THEFT OR LARCEN", "UNAUTHORIZED US", "VANDALISM",
}

# Durham
DURHAM_VIOLENT = {
    "Assault Aggravated", "Assault Intimidation", "Assault Simple",
    "Domestic Disturbance", "Homicide-murder/non-negligent",
    "Homicide-negligent manslaughter", "Kidnap/Abduction",
    "Robbery - Armed", "Robbery - Strong Arm",
    "Sex Offense", "Sex Offense - Assault w/Object", "Sex Offense - Forcible Rape",
    "Sex Offense - Forcible Sodomy", "Sex Offense-forcible fondling",
    "Sex Offense-statutory rape", "Stalking", "Weapons Violations",
}

DURHAM_PROPERTY = {
    "Arson", "Burglary", "Computer Crime", "Embezzelment", "Extortion/Blackmail",
    "Forgery/Counterfeiting", "Fraud-credit card", "Fraud-false pretense",
    "Fraud-impersonation", "Fraud-welfare", "Fraud-wire fraud",
    "Larceny - All Other", "Larceny - From Coin Oper Device",
    "Larceny - From Motor Vehicle", "Larceny - Of Veh Parts/Access",
    "Larceny - Pocket Picking", "Larceny - Purse Snatching",
    "Larceny - Shoplifitng", "Larceny - Theft from Building",
    "Motor Vehicle Theft", "Stolen Property Offenses",
    "Unauthorized Use of Motor Veh", "Vandalism",
}

# Chicago (Primary Type field)
CHICAGO_VIOLENT = {
    "ASSAULT", "BATTERY", "CRIM SEXUAL ASSAULT", "CRIMINAL SEXUAL ASSAULT",
    "HOMICIDE", "HUMAN TRAFFICKING", "INTIMIDATION", "KIDNAPPING",
    "MURDER", "OFFENSE INVOLVING CHILDREN", "ROBBERY", "SEX OFFENSE",
    "STALKING", "WEAPONS VIOLATION",
}

CHICAGO_PROPERTY = {
    "ARSON", "BURGLARY", "CRIMINAL DAMAGE", "CRIMINAL TRESPASS",
    "DECEPTIVE PRACTICE", "MOTOR VEHICLE THEFT", "THEFT",
}

# --- NYC (LAW_CAT_CD field) ---
NYC_VIOLENT    = {"FELONY", "MISDEMEANOR"}   # LAW_CAT_CD — felonies treated as violent proxy
NYC_PROPERTY   = {"VIOLATION"}               # violations treated as property proxy
# More precise mapping using OFNS_DESC handled below
NYC_VIOLENT_OFFENSES = {
    "MURDER & NON-NEGL. MANSLAUGHTER", "RAPE", "ROBBERY", "FELONY ASSAULT",
    "KIDNAPPING", "KIDNAPPING & RELATED OFFENSES", "SEX CRIMES",
    "HOMICIDE-NEGLIGENT-VEHICLE", "HOMICIDE-NEGLIGENT,UNCLASSIFIED",
    "ASSAULT 3 & RELATED OFFENSES", "DANGEROUS WEAPONS",
    "HARASSMENT 2", "STALKING", "DOMESTIC VIOLENCE",
}

NYC_PROPERTY_OFFENSES = {
    "GRAND LARCENY", "PETIT LARCENY", "BURGLARY", "GRAND LARCENY OF MOTOR VEHICLE",
    "THEFT OF SERVICES", "THEFT-FRAUD", "ARSON", "CRIMINAL MISCHIEF & RELATED OF",
    "FRAUDS", "FORGERY", "FRAUD", "STOLEN PROPERTY", "VEHICLE AND TRAFFIC LAWS",
    "UNAUTHORIZED USE OF A VEHICLE", "POSSESSION OF STOLEN PROPERTY",
}


In [12]:
def classify_offense(val, violent_set, property_set):
    if pd.isna(val):
        return "other"
    v = str(val).strip().upper()
    # Check against sets (case-insensitive)
    if val.strip() in violent_set or v in {x.upper() for x in violent_set}:
        return "violent"
    if val.strip() in property_set or v in {x.upper() for x in property_set}:
        return "property"
    return "other"


def make_daily(df, date_col, city_name, violent_set, property_set, offense_col):
    df[date_col] = pd.to_datetime(df[date_col], utc=True, errors="coerce")
    df = df.dropna(subset=[date_col])
    df["date"] = df[date_col].dt.normalize().dt.tz_localize(None)
    df = df[(df["date"] >= START_DATE) & (df["date"] <= END_DATE)]

    df["offense_cat"] = df[offense_col].apply(
        lambda x: classify_offense(x, violent_set, property_set)
    )

    daily = (
        df.groupby("date")
        .agg(
            total_crime_count=("date", "count"),
            violent_count=("offense_cat", lambda x: (x == "violent").sum()),
            property_count=("offense_cat", lambda x: (x == "property").sum()),
            other_count=("offense_cat", lambda x: (x == "other").sum()),
        )
        .reset_index()
    )

    # Fill missing days with 0
    all_days = pd.date_range(START_DATE, END_DATE, freq="D")
    daily = daily.set_index("date").reindex(all_days, fill_value=0).reset_index()
    daily.rename(columns={"index": "date"}, inplace=True)
    daily.insert(0, "city", city_name)
    return daily

In [13]:
# PROCESS CHARLOTTESVILLE

print("Processing Charlottesville")
cville_raw = pd.read_csv("C:/Users/qtg4ys/Documents/DS3001/DS4002/Crime_Data.csv",
                         usecols=["DateReported", "Offense"])
cville_daily = make_daily(cville_raw, "DateReported", "Charlottesville",
                          CVILLE_VIOLENT, CVILLE_PROPERTY, "Offense")
print(f"  {len(cville_daily)} days")

Processing Charlottesville
  1461 days


In [14]:
#looking into cville offenses
print(cville_raw["Offense"].value_counts().to_string())

Offense
Hit and Run                        2319
Assault Simple                     2066
Suspicious Activity                1692
Larceny - All Other                1631
Vandalism                          1600
Larceny - From Motor Vehicle       1585
Assist Citizen - Mental/TDO/ECO    1503
Lost/FoundProperty                 1245
Larceny - Theft from Building      1147
Larceny - Shoplifitng               870
Motor Vehicle Theft                 668
Burglary                            578
Disorderly Conduct                  477
Assault Aggravated                  450
Larceny - Of Veh Parts/Access       440
Misc - Criminal Call                431
Driving Under the Influence         413
Domestic Disturbance                378
Fraud-false pretense                357
Animal Complaint                    352
Shots Fired/Illegal Hunting         334
Death Investigation - DOA           312
Trespass                            293
Assault Intimidation                289
Harassment                      

In [15]:
# PROCESS DURHAM 

print("Processing Durham")
durham_raw = pd.read_csv("C:/Users/qtg4ys/Documents/DS3001/DS4002/City_Crime_(External_Use).csv",
                         usecols=["DATE_REPT", "REPORTEDAS"])
durham_daily = make_daily(durham_raw, "DATE_REPT", "Durham",
                          DURHAM_VIOLENT, DURHAM_PROPERTY, "REPORTEDAS")
print(f"{len(durham_daily)} days")


Processing Durham
1461 days


In [16]:
#Looking into recorded counts
print(durham_raw["REPORTEDAS"].value_counts().to_string())

REPORTEDAS
LARCENY            14225
PRIVATE TOW        13401
BREAK IN OR LAR    13210
LARCENY OF VEHI     5737
COR - DISTURBAN     5313
FRAUD               4887
DAMAGE TO PROPE     4590
COR - DOMESTIC      4145
BREAK IN            3975
VEHICLE STOP        2955
ASSAULT             2520
LOST OR FOUND P     2213
LARCENY IN PROG     2201
COR - HARASSMEN     2188
LARCENY - SHOPL     1559
CARDIAC OR RESP     1361
COR - DOMESTIC      1201
FOLLOW UP           1182
MISSING PERSON      1118
SUSPICIOUS PERS     1056
SEXUAL ASSAULT      1019
RUNAWAY              917
ASSIST PERSON        916
WANTED PERSON        887
SUSPICIOUS VEHI      884
SUSPICIOUS  ACT      882
DOMESTIC VIOLEN      852
ARMED ROBBERY        851
SHOPLIFTER           851
WARRANT OR SUBP      830
NOTIFY POLICE        800
RECOVERED VEHIC      747
ABANDONED VEHIC      664
SOUND OF SHOTS       654
CRT - TRESPASS       650
DISTURBANCE          635
VIOLATION OF OR      634
BREAK IN IN PRO      613
UNKNOWN PROBLEM      612
STOLEN VEHICLE

In [17]:
# PROCESS CHICAGO

print("Processing Chicago")
chicago_raw = pd.read_csv("C:/Users/qtg4ys/Documents/DS3001/DS4002/Crimes_-_2001_to_Present_20260311.csv", usecols=["Date", "Primary Type"])
chicago_daily = make_daily(chicago_raw, "Date", "Chicago",
                           CHICAGO_VIOLENT, CHICAGO_PROPERTY, "Primary Type")
print(f" {len(chicago_daily)} days")


Processing Chicago


C:\Users\qtg4ys\AppData\Local\Temp\ipykernel_62612\2932988209.py:14: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[date_col] = pd.to_datetime(df[date_col], utc=True, errors="coerce")


 1461 days


In [18]:
# PROCESS NYC

print("Processing NYC")
nyc_historic = pd.read_csv("C:/Users/qtg4ys/Documents/DS3001/DS4002/NYPD_Complaint_Data_Historic_20260311 (1).csv", usecols=["CMPLNT_FR_DT", "OFNS_DESC"]) 
nyc_current  = pd.read_csv("C:/Users/qtg4ys/Documents/DS3001/DS4002/NYPD_Complaint_Data_Current_(Year_To_Date)_20260311.csv",  usecols=["CMPLNT_FR_DT", "OFNS_DESC"])

nyc_raw = pd.concat([nyc_historic, nyc_current], ignore_index=True)
nyc_daily = make_daily(nyc_raw, "CMPLNT_FR_DT", "NYC",
                       NYC_VIOLENT_OFFENSES, NYC_PROPERTY_OFFENSES, "OFNS_DESC")
print(f" {len(nyc_daily)} days")


Processing NYC
 1461 days


In [19]:
# COMBINE ALL CITIES

print("\nCombining cities")
frames = [cville_daily, durham_daily, chicago_daily, nyc_daily]

combined = pd.concat(frames, ignore_index=True)
combined = combined.sort_values(["city", "date"]).reset_index(drop=True)


Combining cities


In [20]:
# SAVE CRIME ONLY OUTPUT

output_path = "C:/Users/qtg4ys/Documents/DS3001/DS4002/Combined_Crime_Dataset.csv"
combined.to_csv(output_path, index=False)
print(f"\nSaved to {output_path}")
print(f"Shape: {combined.shape}")
print(f"\nCities in dataset:")
print(combined.groupby("city")[["total_crime_count", "violent_count",
                                 "property_count", "other_count"]].sum())
print(f"\nDate range: {combined['date'].min()} → {combined['date'].max()}")
print(f"\nSample rows:")
print(combined.head(10).to_string())


Saved to C:/Users/qtg4ys/Documents/DS3001/DS4002/Combined_Crime_Dataset.csv
Shape: (5844, 6)

Cities in dataset:
                 total_crime_count  violent_count  property_count  other_count
city                                                                          
Charlottesville              21474             28            1731        19715
Chicago                     998964         352329          549637        96998
Durham                      119542            184             305       119053
NYC                        2216062         488799         1052119       675144

Date range: 2022-01-01 00:00:00 → 2025-12-31 00:00:00

Sample rows:
              city       date  total_crime_count  violent_count  property_count  other_count
0  Charlottesville 2022-01-01                 15              0               2           13
1  Charlottesville 2022-01-02                 11              0               2            9
2  Charlottesville 2022-01-03                  8              0 

In [21]:
# Define Chicago coordinates and the long-term date range
url = "https://archive-api.open-meteo.com/v1/archive"
params = {
    "latitude": 41.85,
    "longitude": -87.65,
    "start_date": "2022-01-01",
    "end_date": "2025-12-31",
    "daily": "temperature_2m_mean,precipitation_sum"
}

response = requests.get(url, params=params)
data = response.json()

# Convert the results into a readable table
df = pd.DataFrame(data['daily'])
print(df.head())

         time  temperature_2m_mean  precipitation_sum
0  2022-01-01                  2.7                5.5
1  2022-01-02                 -3.2                7.0
2  2022-01-03                -11.3                0.0
3  2022-01-04                 -4.4                0.0
4  2022-01-05                 -5.2                0.1


In [23]:
#Add new column header 
import pandas as pd 
import requests

#csv_file_path = "C:/Users/qtg4ys/Documents/DS3001/DS4002/Combined_Crime_Dataset.csv"
crime_df = combined

crime_df["Daily Temperature Mean"] = ""
crime_df["Daily Precipitation Sum"] = ""

#Create map for coordinates and cities 
crime_df["date"] = pd.to_datetime(crime_df["date"]).dt.date.astype(str)

map_coord = {
    "Chicago": (41.85, -87.65),
    "NYC": (40.71, -74.00),
    "Durham": (35.99, -78.90),
    "Charlottesville": (38.03, -78.47)
}

weather_frames = []

url = "https://archive-api.open-meteo.com/v1/archive"

for city, (lat, lon) in map_coord.items():
    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": "2022-01-01",
        "end_date": "2025-12-31",
        "daily": "temperature_2m_mean,precipitation_sum",
        "timezone": "America/New_York"
    }

    response = requests.get(url, params=params)
    response.raise_for_status()
    data = response.json()

    city_weather = pd.DataFrame(data["daily"])
    city_weather["city"] = city

    city_weather = city_weather.rename(columns={
        "time": "date",
        "temperature_2m_mean": "Daily Temperature Mean",
        "precipitation_sum": "Daily Precipitation Sum"
    })

    weather_frames.append(city_weather)

#Combine all city weather data
weather_df = pd.concat(weather_frames, ignore_index=True)

#Merge into crime dataset using city + date
merged_df = crime_df.drop(columns=["Daily Temperature Mean", "Daily Precipitation Sum"], errors="ignore").merge(
    weather_df[["date", "city", "Daily Temperature Mean", "Daily Precipitation Sum"]],
    on=["date", "city"],
    how="left"
)

#Save final file
merged_df.to_csv("weather_crime_final.csv", index=False)

print(merged_df.head())
print("Saved to weather_crime_final.csv")

              city        date  total_crime_count  violent_count  \
0  Charlottesville  2022-01-01                 15              0   
1  Charlottesville  2022-01-02                 11              0   
2  Charlottesville  2022-01-03                  8              0   
3  Charlottesville  2022-01-04                 16              0   
4  Charlottesville  2022-01-05                 12              0   

   property_count  other_count  Daily Temperature Mean  \
0               2           13                    14.7   
1               2            9                    15.4   
2               1            7                     0.9   
3               1           15                    -6.4   
4               0           12                     0.2   

   Daily Precipitation Sum  
0                      9.0  
1                     16.9  
2                     29.4  
3                      0.0  
4                      0.0  
Saved to weather_crime_final.csv
